In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")


@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

/tmp/ipykernel_174916/4208644247.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [3]:
from dataclasses import dataclass

@dataclass
class UserRole:
    user_role: str = "external"

In [4]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role
    
    if user_role == "internal":
        pass # internal users get access to all tools
    else:
        tools = [web_search] # external users only get access to web search
        request = request.override(tools=tools) 

    return handler(request)

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "external"}
)

print(response["messages"][-1].content)

I don’t have access to your database, so I can’t see the exact number. Could you tell me:

- What database/system are you using (SQL, MongoDB, etc.)?
- How do you define an “artist” (is it a row in a table, a document in a collection, etc.)?
- Do you want the total rows, or distinct artists (by id or by name), and should we include only active artists or apply other filters?

If you just want the general queries:

SQL (e.g., PostgreSQL/MySQL)
- Total rows: SELECT COUNT(*) AS artist_count FROM artists;
- Distinct by id (if artist_id is the key): SELECT COUNT(DISTINCT artist_id) AS artist_count FROM artists;
- Distinct by name: SELECT COUNT(DISTINCT name) AS artist_count FROM artists;
- Active artists only: SELECT COUNT(*) AS artist_count FROM artists WHERE status = 'active';

MongoDB
- Total documents: db.artists.countDocuments({});
- Active artists only: db.artists.countDocuments({ status: 'active' });

If you share your schema or paste a small sample, I’ll tailor the exact query and c